# 01 — Exploración de la API del Ministerio para la Transición Ecológica

**Proyecto:** Optimización del repostaje en flotas comerciales mediante predicción de precios de carburantes  
**Autor:** Víctor González Martín  
**Notebook:** 01 — Exploración inicial de la API

## Objetivo del notebook

Explorar la API REST oficial del Ministerio para la Transición Ecológica que publica los precios de los carburantes en las estaciones de servicio españolas. Antes de la descarga masiva del histórico, se valida la estructura de los datos, los campos disponibles y el comportamiento de los endpoints.

## Endpoints documentados

- **Datos actuales:** `https://sedeaplicaciones.minetur.gob.es/ServiciosRESTCarburantes/PreciosCarburantes/EstacionesTerrestres/`
- **Datos históricos por fecha:** `https://sedeaplicaciones.minetur.gob.es/ServiciosRESTCarburantes/PreciosCarburantes/EstacionesTerrestresHist/{FECHA}` con formato de fecha `DD-MM-AAAA`.

## Salida esperada

Conocimiento estructurado de la API: qué campos tiene, qué nombres usan, qué tipos de datos devuelven, qué peculiaridades hay que tener en cuenta para la descarga sistemática del histórico.

## Ficha técnica del dataset utilizado

| Concepto | Valor |
|---|---|
| **Fuente** | API REST oficial del Ministerio para la Transición Ecológica |
| **URL** | https://sedeaplicaciones.minetur.gob.es/ServiciosRESTCarburantes/PreciosCarburantes/EstacionesTerrestres/ |
| **Tipo de consulta** | Endpoint de datos actuales (snapshot) |
| **Fecha de descarga** | 06/06/2026 |
| **Modo de obtención** | Descarga vía PowerShell (`Invoke-WebRequest`) ante problemas SSL del servidor desde Python |
| **Formato origen** | JSON UTF-8 |
| **Tamaño del fichero** | ~11 MB |
| **Almacenamiento local** | `data/raw/precios_actuales_06-06-2026.json` |

**Nota sobre reproducibilidad**: el presente notebook trabaja sobre un snapshot estático del dataset. En la fase posterior del TFM (descarga histórica) se construirá una serie temporal de ~2 años mediante consultas sistemáticas al endpoint histórico de la API.

In [1]:
# Verificación del entorno de ejecución
# Esta celda confirma que el notebook se ejecuta en el entorno conda correcto del TFM.

import sys
import os

# Configuración del entorno esperado
ENTORNO_ESPERADO = "repostapro"
PROYECTO = "RepostaPro"

print(f"Proyecto: {PROYECTO}")
print(f"Python ejecutable: {sys.executable}")
print(f"Versión de Python: {sys.version.split()[0]}")
print(f"Directorio de trabajo: {os.getcwd()}")

# Comprobación explícita
if ENTORNO_ESPERADO in sys.executable:
    print(f"\nEntorno conda correcto: {ENTORNO_ESPERADO}")
else:
    print(f"\nATENCIÓN: NO estás en el entorno {ENTORNO_ESPERADO}. Reinicia el kernel.")

Proyecto: RepostaPro
Python ejecutable: C:\Users\vicgon\AppData\Local\anaconda3\envs\repostapro\python.exe
Versión de Python: 3.12.12
Directorio de trabajo: C:\TFM\notebooks

Entorno conda correcto: repostapro


In [2]:
# Imports necesarios para la exploración
import requests
import pandas as pd
from pprint import pprint
import json

# Configuración de visualización para que pandas no recorte columnas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

# Versiones de las librerías (útil para reproducibilidad del codigo)
print(f"requests: {requests.__version__}")
print(f"pandas: {pd.__version__}")

requests: 2.34.2
pandas: 3.0.3


In [3]:
# Carga de datos desde fichero local
# Nota técnica: durante el desarrollo se detectaron caídas SSL intermitentes del servidor del Ministerio que rompían la conexión TLS desde Python (SSLEOFError), con un patrón temporal asociado a horarios nocturnos.
# Para garantizar reproducibilidad y continuidad del trabajo, los datos se descargaron una sola vez desde el navegador y se almacenaron en local.
# El procesamiento posterior es idéntico al que se obtendría directamente de la API.
#Origen: https://sedeaplicaciones.minetur.gob.es/ServiciosRESTCarburantes/PreciosCarburantes/EstacionesTerrestres   Fecha de descarga: 06/06/2026

import json
from pathlib import Path

ARCHIVO_JSON = Path("../data/raw/precios_actuales_06-06-2026.json")

print(f"Leyendo datos desde: {ARCHIVO_JSON.name}")

with open(ARCHIVO_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

# Información sobre lo cargado
print(f"Resultado de la consulta: {data.get('ResultadoConsulta', 'desconocido')}")
print(f"Fecha de los datos:       {data.get('Fecha', 'desconocida')}")
print(f"Número de estaciones:     {len(data.get('ListaEESSPrecio', [])):,}")
print(f"Tamaño del fichero:       {ARCHIVO_JSON.stat().st_size / 1024 / 1024:.2f} MB")

Leyendo datos desde: precios_actuales_06-06-2026.json
Resultado de la consulta: OK
Fecha de los datos:       06/06/2026 15:16:06
Número de estaciones:     11,475
Tamaño del fichero:       11.64 MB


### Código alternativo: descarga directa desde la API

Como referencia técnica, el código original que conecta directamente con la API REST del Ministerio para la Transición Ecológica sería el siguiente:

```python
# import requests
# 
# URL_ACTUAL = "https://sedeaplicaciones.minetur.gob.es/ServiciosRESTCarburantes/PreciosCarburantes/EstacionesTerrestres/"
# HEADERS = {"Accept": "application/json"}
# 
# response = requests.get(URL_ACTUAL, headers=HEADERS, timeout=30)
# data = response.json()
```

Este código funciona correctamente cuando el servidor del Ministerio responde sin problemas TLS. Para el desarrollo del TFM se ha optado por trabajar sobre fichero local descargado una sola vez, por estabilidad y reproducibilidad. En la fase de actualización del dataset (Sprint 2 — descarga histórica), se implementará lógica de reintentos para manejar las caídas intermitentes del servidor.

In [6]:
# Exploración de la estructura de la respuesta JSON
# La variable 'data' ya se cargó en la celda anterior desde el fichero local.
# Aquí se inspeccionan las claves de primer nivel y su contenido resumido.

print("Claves de primer nivel en la respuesta JSON:")
for clave in data.keys():
    valor = data[clave]
    # Mostramos el tipo y, si es una lista, su longitud
    if isinstance(valor, list):
        print(f"  • {clave!r:35} → lista con {len(valor):,} elementos")
    elif isinstance(valor, dict):
        print(f"  • {clave!r:35} → diccionario con {len(valor)} claves")
    else:
        # Si es un valor simple (string, número), lo mostramos truncado
        valor_str = str(valor)[:80]
        print(f"  • {clave!r:35} → {valor_str!r}")

Claves de primer nivel en la respuesta JSON:
  • 'Fecha'                             → '06/06/2026 15:16:06'
  • 'ListaEESSPrecio'                   → lista con 11,475 elementos
  • 'Nota'                              → 'Archivo de todos los productos en todas las estaciones de servicio. La actualiza'
  • 'ResultadoConsulta'                 → 'OK'


In [7]:
# ──────────────────────────────────────────────────────────────────────────────
# Explorar la estructura de una estación individual
# ──────────────────────────────────────────────────────────────────────────────

# Extraemos la lista de estaciones del JSON
estaciones = data["ListaEESSPrecio"]

# Cogemos la primera estación como muestra
primera_estacion = estaciones[0]

# Mostramos todos sus campos de forma legible
print(f"Número total de estaciones: {len(estaciones):,}\n")
print(f"Campos disponibles en cada estación ({len(primera_estacion)} campos):\n")

# Recorremos los campos y los mostramos en columnas alineadas
for campo, valor in primera_estacion.items():
    # Truncamos el valor si es muy largo, para mantener legibilidad
    valor_str = str(valor)
    if len(valor_str) > 60:
        valor_str = valor_str[:57] + "..."
    print(f"  {campo:30} → {valor_str!r}")

Número total de estaciones: 11,475

Campos disponibles en cada estación (41 campos):

  C.P.                           → '02250'
  Dirección                      → 'AVENIDA CASTILLA LA MANCHA, 26'
  Horario                        → 'L-D: 07:00-22:00'
  Latitud                        → '39,211417'
  Localidad                      → 'ABENGIBRE'
  Longitud (WGS84)               → '-1,539167'
  Margen                         → 'D'
  Municipio                      → 'Abengibre'
  Precio Adblue                  → ''
  Precio Amoniaco                → ''
  Precio Biodiesel               → ''
  Precio Bioetanol               → ''
  Precio Biogas Natural Comprimido → ''
  Precio Biogas Natural Licuado  → ''
  Precio Diésel Renovable        → ''
  Precio Gas Natural Comprimido  → ''
  Precio Gas Natural Licuado     → ''
  Precio Gases licuados del petróleo → ''
  Precio Gasoleo A               → '1,559'
  Precio Gasoleo B               → '1,129'
  Precio Gasoleo Premium         → ''
  Precio Gas

## Estructura de la respuesta de la API

### Características generales
- La API devuelve un JSON con **4 claves de primer nivel**: `Fecha`, `ListaEESSPrecio`, `Nota`, `ResultadoConsulta`.
- La respuesta incluye la lista completa de estaciones de servicio activas en España en el momento de la consulta.
- Cada estación contiene **41 campos**.
- Codificación: UTF-8 (soporte completo de caracteres españoles).
- El campo `Fecha` permite trazar el momento exacto de la consulta, lo que es fundamental para construir el histórico.

### Categorías de campos por estación

**Identificación (5 campos):** `IDEESS` (clave primaria), `IDMunicipio`, `IDProvincia`, `IDCCAA`, `C.P.`

**Ubicación (6 campos):** `Dirección`, `Localidad`, `Municipio`, `Provincia`, `Latitud`, `Longitud (WGS84)`

**Descriptivos (5 campos):** `Rótulo` (marca comercial), `Horario`, `Margen`, `Tipo Venta`, `Remisión`

**Precios (22 campos):** carburantes convencionales (Gasóleo A, Gasóleo B, Gasolina 95 E5, etc.) y alternativos (GLP, GNC, GNL, hidrógeno, bioetanol, biodiésel, etc.).

**Composición química (2 campos):** `% BioEtanol`, `% Éster metílico`.

### Particularidades técnicas detectadas

1. **Decimales con coma**: precios y coordenadas se entregan como strings con coma decimal (ej. `'1,559'`). Requerirá conversión a `float` reemplazando `,` por `.`.
2. **Valores ausentes**: los precios no disponibles se entregan como string vacío (`''`), no como `null`. Deberán convertirse a `NaN` para análisis estadístico correcto.
3. **Códigos con ceros a la izquierda**: identificadores administrativos (`IDProvincia`, `C.P.`) deben mantenerse como strings para preservar los ceros.
4. **Cobertura de carburantes heterogénea**: las estaciones rurales suelen ofrecer solo 2-3 carburantes; las urbanas pueden ofrecer la gama completa.

### Carburantes incluidos en el análisis

Para el alcance del TFM (optimización de repostaje en flotas comerciales) se trabajará con:

**Núcleo del análisis predictivo:**
- **Gasóleo A** — combustible mayoritario en flotas de transporte y mercancías.
- **Gasóleo Premium** — variante premium del diésel, para análisis comparativo.
- **Gasolina 95 E5** — combustible mayoritario en turismos comerciales.
- **Gasolina 98 E5** — variante premium de gasolina, para análisis comparativo.

**Análisis secundario (descriptivo y geográfico):**
- **GLP** (Gases Licuados del Petróleo) — análisis de cobertura y diferencial de precio respecto al diésel, sin modelado predictivo debido a la menor cobertura geográfica.

**Excluidos del análisis:** Gasóleo B (carburante agrícola subsidiado, fuera del alcance comercial), Biodiésel, GNC, GNL, hidrógeno, metanol y otros minoritarios con cobertura insuficiente para análisis estadístico.

In [8]:
# Conversión a DataFrame de pandas
# La lista de estaciones del JSON se convierte a un DataFrame de pandas para facilitar el análisis exploratorio. pd.json_normalize aplana automáticamente la lista de diccionarios en una tabla tabular.
# Extraemos la lista de estaciones del JSON
estaciones = data["ListaEESSPrecio"]

# Convertimos a DataFrame
df = pd.json_normalize(estaciones)

# Información básica del DataFrame resultante
print(f"Dimensiones del DataFrame: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Memoria ocupada:           {df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")
print(f"\nPrimeras filas del DataFrame:")
df.head(3)

Dimensiones del DataFrame: 11,475 filas × 41 columnas
Memoria ocupada:           5.19 MB

Primeras filas del DataFrame:


,C.P.,Dirección,Horario,Latitud,Localidad,Longitud (WGS84),Margen,Municipio,Precio Adblue,Precio Amoniaco,Precio Biodiesel,Precio Bioetanol,Precio Biogas Natural Comprimido,Precio Biogas Natural Licuado,Precio Diésel Renovable,Precio Gas Natural Comprimido,Precio Gas Natural Licuado,Precio Gases licuados del petróleo,Precio Gasoleo A,Precio Gasoleo B,Precio Gasoleo Premium,Precio Gasolina 95 E10,Precio Gasolina 95 E25,Precio Gasolina 95 E5,Precio Gasolina 95 E5 Premium,Precio Gasolina 95 E85,Precio Gasolina 98 E10,Precio Gasolina 98 E5,Precio Gasolina Renovable,Precio Hidrogeno,Precio Metanol,Provincia,Remisión,Rótulo,Tipo Venta,% BioEtanol,% Éster metílico,IDEESS,IDMunicipio,IDProvincia,IDCCAA
0,02250,"AVENIDA CASTILLA LA MANCHA, 26",L-D: 07:00-22:00,"39,211417",ABENGIBRE,"-1,539167",D,Abengibre,,,,,,,,,,,"1,559","1,129",,,,"1,439",,,,,,,,ALBACETE,dm,Nº 10.935,P,"0,0","0,0",4375,52,02,07
1,02152,"CR CM-332, 46,4",L-D: 7:00-23:00,"39,100389",ALATOZ,"-1,346083",I,Alatoz,,,,,,,,,,,"1,879",,"1,969",,,"1,649",,,,,,,,ALBACETE,dm,REPSOL,P,"0,0","0,0",5122,53,02,07
2,02001,CALLE PRINCIPE DE ASTURIAS (POLÍGONO DE ROMICA...,L-D: 06:00-22:00,"39,054694",ALBACETE,"-1,832000",I,Albacete,"1,119",,,,,,,,,,"1,699",,"1,789",,,"1,539",,,,"1,679",,,,ALBACETE,OM,BP ROMICA,P,"0,0","0,0",12054,54,02,07


In [9]:
# Tipos de datos y valores ausentes
# Inspección formal de los tipos de cada columna y conteo de valores ausentes.
# pd.json_normalize ha cargado todo como 'object' (texto) porque la API devuelve strings para todos los campos. Esto es esperable y se corregirá en el preprocesado.

print("Información general del DataFrame:")
print("=" * 70)
df.info(verbose=True, show_counts=True)

Información general del DataFrame:
<class 'pandas.DataFrame'>
RangeIndex: 11475 entries, 0 to 11474
Data columns (total 41 columns):
 #   Column                              Non-Null Count  Dtype
---  ------                              --------------  -----
 0   C.P.                                11475 non-null  str  
 1   Dirección                           11475 non-null  str  
 2   Horario                             11475 non-null  str  
 3   Latitud                             11475 non-null  str  
 4   Localidad                           11475 non-null  str  
 5   Longitud (WGS84)                    11475 non-null  str  
 6   Margen                              11475 non-null  str  
 7   Municipio                           11475 non-null  str  
 8   Precio Adblue                       11475 non-null  str  
 9   Precio Amoniaco                     11475 non-null  str  
 10  Precio Biodiesel                    11475 non-null  str  
 11  Precio Bioetanol                    11475 n

## Limpieza preliminar de los datos

La inspección anterior revela tres tareas de limpieza imprescindibles antes de cualquier análisis estadístico:

1. **Conversión de strings vacíos a `NaN`**: la API devuelve cadenas vacías (`""`) en los precios que la estación no oferta. Pandas no las reconoce como valores ausentes por defecto, lo que falsea el conteo de cobertura.

2. **Conversión de precios a tipo numérico**: todos los precios vienen como strings con coma decimal (ej. `"1,559"`). Hay que sustituir la coma por punto y convertir a `float`.

3. **Conversión de coordenadas a tipo numérico**: latitud y longitud presentan el mismo problema que los precios.

Los identificadores administrativos (`C.P.`, `IDProvincia`, etc.) se mantienen como string para preservar los ceros a la izquierda.

In [10]:
# Conversión de strings vacíos a NaN
# La API devuelve "" para los precios que la estación no comercializa.
# Pandas los cuenta como valores presentes, lo que falsea el análisis de cobertura.
# Sustituimos "" por NaN para que el conteo de no-nulos sea correcto.

import numpy as np

df = df.replace("", np.nan)

# Verificación: ahora algunos campos deberían tener valores ausentes detectados
print("Conteo de valores presentes por columna (tras conversión):")
print("=" * 70)
print(df.notna().sum().sort_values(ascending=False))

Conteo de valores presentes por columna (tras conversión):
C.P.                                  11475
Dirección                             11475
Horario                               11475
Latitud                               11475
Localidad                             11475
Longitud (WGS84)                      11475
Margen                                11475
Municipio                             11475
Rótulo                                11475
Remisión                              11475
Tipo Venta                            11475
% BioEtanol                           11475
% Éster metílico                      11475
IDEESS                                11475
IDMunicipio                           11475
IDCCAA                                11475
IDProvincia                           11475
Provincia                             11475
Precio Gasoleo A                      11262
Precio Gasolina 95 E5                 10900
Precio Gasoleo Premium                 5978
Precio Gasolina 9

## Conversión de tipos numéricos

Tras la conversión de cadenas vacías a `NaN`, el siguiente paso es transformar al tipo numérico los campos que efectivamente contienen valores numéricos:

- **Precios de carburantes**: 22 columnas. Vienen como strings con coma decimal (ej. `"1,559"`). Conversión: sustituir `,` por `.` y convertir a `float`.
- **Coordenadas geográficas**: latitud y longitud. Mismo tratamiento.
- **Porcentajes**: `% BioEtanol` y `% Éster metílico`. Mismo tratamiento.

Los identificadores administrativos (`C.P.`, `IDEESS`, `IDProvincia`, `IDMunicipio`, `IDCCAA`) se mantienen como string para preservar los ceros a la izquierda.

## Nota: AdBlue como pata secundaria del análisis

Durante la exploración se detecta que el **AdBlue** está presente en 2.879 estaciones (25,1% del total). Aunque no es un combustible propiamente dicho (es un aditivo de urea para reducir emisiones de NOx en motores diésel Euro 6), su consumo es obligatorio en flotas modernas de transporte. Por tanto, **constituye un componente real del coste operativo de combustible** de cualquier flota Euro 6.

Decisión: se incluirá el AdBlue como **análisis secundario complementario al diésel**, abarcando:

- Análisis de cobertura geográfica conjunta con el Gasóleo A.
- Análisis de correlación de precios AdBlue / Gasóleo A.
- Incorporación opcional del precio AdBlue como variable explicativa en el modelo de Gasóleo A.
- Inclusión del coste del AdBlue en el cálculo de impacto económico para flotas tipo de transporte.

No se entrenará modelo predictivo independiente de AdBlue debido a la cobertura limitada (25%).

In [11]:
# Conversión de precios, coordenadas y porcentajes a tipo numérico
# Los valores vienen como strings con coma decimal española (ej. "1,559").
# Sustituimos la coma por punto y convertimos a float.
# Los NaN se preservan automáticamente durante la conversión.

# Identificar las columnas que hay que convertir
columnas_precio = [col for col in df.columns if col.startswith("Precio")]
columnas_coordenadas = ["Latitud", "Longitud (WGS84)"]
columnas_porcentajes = ["% BioEtanol", "% Éster metílico"]

columnas_a_numerico = columnas_precio + columnas_coordenadas + columnas_porcentajes

print(f"Columnas a convertir a numérico: {len(columnas_a_numerico)}")
print(f"  - Precios:      {len(columnas_precio)}")
print(f"  - Coordenadas:  {len(columnas_coordenadas)}")
print(f"  - Porcentajes:  {len(columnas_porcentajes)}")

# Conversión: reemplazar coma por punto y pasar a float
for col in columnas_a_numerico:
    df[col] = df[col].str.replace(",", ".", regex=False).astype(float)

# Verificación: ahora estas columnas deberían ser float64
print(f"\nVerificación de los tipos tras la conversión:")
print("=" * 70)
print(df.dtypes.value_counts())
print(f"\nEjemplo del 'Precio Gasoleo A':")
print(df["Precio Gasoleo A"].describe())

Columnas a convertir a numérico: 27
  - Precios:      23
  - Coordenadas:  2
  - Porcentajes:  2

Verificación de los tipos tras la conversión:
float64    27
str        14
Name: count, dtype: int64

Ejemplo del 'Precio Gasoleo A':
count    11262.000000
mean         1.617290
std          0.087637
min          1.299000
25%          1.555000
50%          1.639000
75%          1.684000
max          1.999000
Name: Precio Gasoleo A, dtype: float64


## Caracterización de la oferta por estación

¿Cuántos carburantes distintos ofrece cada estación? Este indicador nos permite distinguir entre estaciones "monoproducto" (estaciones rurales con solo diésel y gasolina) y estaciones "premium" con gama completa (urbanas, autopistas, multiservicio).

In [12]:
# Cálculo del número de carburantes ofertados por estación
columnas_precio = [col for col in df.columns if col.startswith("Precio")]

# Contamos cuántos precios NO son NaN por cada fila (estación)
df["n_carburantes"] = df[columnas_precio].notna().sum(axis=1)

# Distribución del número de carburantes por estación
print("Distribución del número de carburantes ofertados por estación:")
print("=" * 70)
print(df["n_carburantes"].describe())
print("\nDistribución detallada:")
print(df["n_carburantes"].value_counts().sort_index())

Distribución del número de carburantes ofertados por estación:
count    11475.000000
mean         3.739434
std          1.231908
min          1.000000
25%          3.000000
50%          4.000000
75%          5.000000
max          9.000000
Name: n_carburantes, dtype: float64

Distribución detallada:
n_carburantes
1     209
2    2096
3    2045
4    4134
5    2219
6     670
7      94
8       7
9       1
Name: count, dtype: int64


### Las estaciones más completas y más exclusivas

Antes de continuar con el análisis general, examinamos los extremos de la distribución por su valor descriptivo del mercado:

- ¿Qué estaciones ofrecen la gama más completa (8-9 carburantes)?
- ¿Qué estaciones se especializan en un solo producto?

In [13]:
# Estaciones con la gama más amplia (8 o 9 carburantes)
print("ESTACIONES CON MAYOR DIVERSIDAD DE CARBURANTES (≥8)")
print("=" * 80)
estaciones_top = df[df["n_carburantes"] >= 8][
    ["n_carburantes", "Rótulo", "Municipio", "Provincia", "Dirección"]
].sort_values("n_carburantes", ascending=False)
print(estaciones_top.to_string(index=False))

# Distribución de las estaciones monoproducto
print("\n" + "=" * 80)
print("ESTACIONES MONOPRODUCTO (1 carburante): rótulos más frecuentes")
print("=" * 80)
mono = df[df["n_carburantes"] == 1]
print(mono["Rótulo"].value_counts().head(10))

ESTACIONES CON MAYOR DIVERSIDAD DE CARBURANTES (≥8)
 n_carburantes               Rótulo                            Municipio Provincia                                          Dirección
             9         ALAS CENTRAL Vandellòs i l'Hospitalet de l'Infant TARRAGONA                              CARRER DE JOAN ORÓ, 2
             8               REPSOL                             Albatera  ALICANTE                  CARRETERA CATRAL-ALBATERA KM. 4,2
             8               REPSOL                 Jerez de la Frontera     CÁDIZ                         CARRETERA A-381 KM. 11,500
             8         BEROIL, S.L.                               Rubena    BURGOS                                         N-I km 249
             8               REPSOL                              Hernani  GIPUZKOA                              AUTOPISTA AP-8 KM. 22
             8 ARENAS CAMACHO, S.L.                             Mengíbar      JAÉN                     N-323A (BAILÉN-MOTRIL) km 14,4
          

### Tipo de venta: estaciones públicas vs restringidas

La API distingue entre estaciones de venta pública (`Tipo Venta = "P"`) y restringida (`Tipo Venta = "R"`). Las estaciones restringidas son normalmente cooperativas agrícolas, estaciones internas de empresas de transporte, depósitos militares, etc. Para el alcance de este TFM (flotas comerciales con acceso libre al mercado), interesa centrarse en las de venta pública.

In [14]:
# Análisis del tipo de venta y su relación con la oferta de carburantes
print("Distribución por tipo de venta:")
print("=" * 70)
print(df["Tipo Venta"].value_counts(dropna=False))
print(f"\nPorcentaje sobre el total:")
print((df["Tipo Venta"].value_counts(normalize=True) * 100).round(2))

print("\n" + "=" * 70)
print("¿Las estaciones monoproducto son mayoritariamente restringidas?")
print("=" * 70)
tabla_cruzada = df.groupby("Tipo Venta")["n_carburantes"].agg(["count", "mean", "median"])
print(tabla_cruzada.round(2))

Distribución por tipo de venta:
Tipo Venta
P    11475
Name: count, dtype: int64

Porcentaje sobre el total:
Tipo Venta
P    100.0
Name: proportion, dtype: float64

¿Las estaciones monoproducto son mayoritariamente restringidas?
            count  mean  median
Tipo Venta                     
P           11475  3.74     4.0


## Nota: AdBlue como pata secundaria del análisis

Durante la exploración se detecta que el **AdBlue** está presente en 2.879 estaciones (25,1% del total). Aunque no es un combustible propiamente dicho (es un aditivo de urea para reducir emisiones de NOx en motores diésel Euro 6), su consumo es obligatorio en flotas modernas de transporte. Por tanto, **constituye un componente real del coste operativo de combustible** de cualquier flota Euro 6.

Decisión: se incluirá el AdBlue como **análisis secundario complementario al diésel**, abarcando:

- Análisis de cobertura geográfica conjunta con el Gasóleo A.
- Análisis de correlación de precios AdBlue / Gasóleo A.
- Incorporación opcional del precio AdBlue como variable explicativa en el modelo de Gasóleo A.
- Inclusión del coste del AdBlue en el cálculo de impacto económico para flotas tipo de transporte.

No se entrenará modelo predictivo independiente de AdBlue debido a la cobertura limitada (25%).

In [ ]:
# Diagnóstico del nuevo archivo
from pathlib import Path

ARCHIVO_JSON = Path("../data/raw/precios_actuales_06-06-2026.json")

with open(ARCHIVO_JSON, "rb") as f:  # rb = bytes, así vemos invisibles
    contenido = f.read()

print(f"Tamaño total: {len(contenido):,} bytes")
print(f"\n--- PRIMEROS 30 BYTES (en raw) ---")
print(contenido[:30])
print(f"\n--- PRIMEROS 30 BYTES (hex) ---")
print(contenido[:30].hex())
print(f"\n--- ÚLTIMOS 30 BYTES ---")
print(contenido[-30:])